# Human Activity Recognition Using Hidden Markov Models

**Formative 2 — HMM**  
**Group/Student:** Individual  

## Overview

This notebook implements a **Hidden Markov Model (HMM) from scratch using NumPy** to infer
hidden human activity states (standing, walking, jumping, still) from accelerometer and
gyroscope sensor data.

### Project tasks covered
1. Data collection — labelled CSV sensor files (accelerometer + gyroscope, timestamps)
2. Feature extraction — time-domain + frequency-domain features, Z-score normalization
3. HMM setup — hidden states, observations, transition (A), emission (B), initial probabilities
4. Model implementation — Viterbi decoding + Baum-Welch training (from scratch)
5. Evaluation on unseen data — sensitivity, specificity, accuracy, confusion matrix
6. Analysis & reflection — visualizations and discussion

## 1. Setup and Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make sure src is on the path
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

# Paths and constants (inlined — no data_generator.py needed with real data)
BASE_DIR = os.path.dirname(os.path.abspath(""))
TRAIN_DIR = os.path.join(BASE_DIR, "data", "train")
TEST_DIR = os.path.join(BASE_DIR, "data", "test")
SAMPLING_RATE_HZ = 50
ACTIVITIES = ["standing", "walking", "jumping", "still"]

from features import extract_all_features_from_dir, z_score_normalize
from hmm import DiscreteHMM
from visualization import (
    plot_transition_matrix,
    plot_emission_matrix,
    plot_decoded_sequence,
    plot_confusion_matrix,
    plot_sample_signals,
    plot_log_likelihood,
    compute_metrics,
)

from collections import Counter

print("Imports successful.")
print(f"Activities: {ACTIVITIES}")
print(f"Sampling rate: {SAMPLING_RATE_HZ} Hz")

## 2. Data Collection

Generate (or load) labelled sensor CSV files. Each file contains:
`timestamp, accel_x, accel_y, accel_z, gyro_x, gyro_y, gyro_z` with a metadata header
recording the sampling rate and activity label.

For this individual submission we use **19 labelled real recordings across 4 activities** (11 training, 8 unseen test), each ~8–11 seconds long at ~62.5 Hz.

> In a real submission you would replace this step with your own recorded Sensor Logger
> data and include links to 4 data-collection videos (one per activity).

In [ ]:
# Generate dataset (DO NOT RUN with real data — this would overwrite real CSVs)
# Real data is already prepared via convert_real_data.py
# generate_dataset()

# Count files
for split in ["train", "test"]:
    split_dir = os.path.join(os.getcwd(), "data", split)
    total = 0
    for act in ACTIVITIES:
        n = len([f for f in os.listdir(os.path.join(split_dir, act)) if f.endswith(".csv")])
        total += n
        print(f"  {split}/{act}: {n} files")
    print(f"  {split} total: {total} files\n")

### Sample raw signals

In [ ]:
plot_sample_signals(TRAIN_DIR, activities=ACTIVITIES, save_path="figures/sample_signals.png", fs=SAMPLING_RATE_HZ)

## 3. Feature Extraction

From each 2-second sliding window we compute:

| Feature | Domain | Why it helps distinguish activities |
|---------|--------|-------------------------------------|
| SMA (Signal Magnitude Area) | Time | Overall movement intensity — high for jumping/walking, low for still/standing |
| Mean acceleration | Time | Average movement magnitude |
| Variance of acceleration | Time | Variability — jumping has high variance, still has near-zero |
| Dominant frequency (FFT) | Frequency | Walking/jumping have clear periodic frequencies; still/standing near 0 Hz |
| Spectral energy (FFT) | Frequency | Concentration of energy — periodic activities concentrate energy at few frequencies |

Features are **Z-score normalized** using statistics computed on the training set only,
then applied identically to the unseen test set.

In [ ]:
WINDOW_SEC = 2.0
N_OBS = 20
N_HIDDEN = 4

# --- Training features ---
X_train_raw, y_train, train_info, feature_names = extract_all_features_from_dir(
    TRAIN_DIR, window_sec=WINDOW_SEC
)
print(f"Training windows: {X_train_raw.shape[0]}")
print(f"Feature names: {feature_names}")
print(f"Files per activity: {{a: sum(1 for i in train_info if i['activity']==a) for a in ACTIVITIES}}")

# --- Z-score normalization (fit on train) ---
X_train, mean, std = z_score_normalize(X_train_raw)
print(f"\nZ-score normalization applied.")
print(f"Mean: {np.round(mean, 3)}")
print(f"Std:  {np.round(std, 3)}")

# --- Discretise to observation symbols ---
X_train_symbols, bin_edges = DiscreteHMM.encode_observations(X_train, n_obs=N_OBS, random_state=42)
train_lengths = [info["n_windows"] for info in train_info]
print(f"\nDiscretised to {N_OBS} observation symbols.")
print(f"Number of training sequences (files): {len(train_lengths)}")

## 4. HMM Setup & Implementation

### Model components

| Element | Description |
|---------|-------------|
| Hidden states (Z) | 4 activities: standing, walking, jumping, still |
| Observations (X) | Discretised feature vectors (20 symbols) |
| Transition (A) | 4×4 matrix — probability of switching activity between windows |
| Emission (B) | 4×20 matrix — probability of each observation symbol per activity |
| Initial (π) | Start probability for each activity |

### Algorithms
- **Baum-Welch (EM)** — learns A, B, π from unlabelled observation sequences; stops when log-likelihood improvement < ε (1e-4).
- **Viterbi decoding** — finds the most likely hidden state sequence given observations.

Both implemented from scratch in NumPy in `src/hmm.py`.

In [ ]:
print("Training HMM with Baum-Welch (from scratch)...")
model = DiscreteHMM(n_hidden=N_HIDDEN, n_obs=N_OBS, random_state=42)
model.fit(
    X_train_symbols,
    lengths=train_lengths,
    n_iter=200,
    tol=1e-4,
    verbose=True,
)
print(f"\nTraining complete.")
print(f"Converged: {model.converged_}")
print(f"EM iterations: {model.n_iter_}")

## 5. Evaluation on Unseen Data

Test on **new recordings not used in training** (2 files per activity).

Metrics reported: sensitivity, specificity, and overall accuracy per activity, plus a
confusion matrix. State-to-activity mapping is derived from the most frequent true label
assigned to each hidden state during training.

In [ ]:
# --- Test features (apply train mean/std, no re-fit) ---
X_test_raw, y_test, test_info, _ = extract_all_features_from_dir(TEST_DIR, window_sec=WINDOW_SEC)
X_test, _, _ = z_score_normalize(X_test_raw, mean=mean, std=std)
X_test_symbols, _ = DiscreteHMM.encode_observations(X_test, n_obs=N_OBS, random_state=42)
test_lengths = [info["n_windows"] for info in test_info]

# --- State -> activity mapping from training ---
state_to_activity = {}
for s in range(N_HIDDEN):
    mask = model.predict(X_train_symbols) == s
    if mask.sum() > 0:
        state_to_activity[s] = Counter(y_train[mask]).most_common(1)[0][0]
    else:
        state_to_activity[s] = ACTIVITIES[s]

print("State -> activity mapping:", state_to_activity)

# --- Predict on each test sequence ---
y_test_pred = []
y_test_true = []
seq_idx = 0
for info in test_info:
    n_win = info["n_windows"]
    seq_symbols = X_test_symbols[seq_idx:seq_idx + n_win]
    pred_states = model.predict(seq_symbols)
    pred_activities = [state_to_activity[s] for s in pred_states]
    y_test_pred.extend(pred_activities)
    y_test_true.extend([info["activity"]] * n_win)
    seq_idx += n_win

y_test_pred = np.array(y_test_pred)
y_test_true = np.array(y_test_true)
print(f"\nTest windows predicted: {len(y_test_pred)}")

In [ ]:
# --- Metrics ---
metrics = compute_metrics(y_test_true, y_test_pred, activities=ACTIVITIES)

print("Evaluation results (unseen test data):")
print(f"{'Activity':<12} {'Sensitivity':>12} {'Specificity':>12} {'Support':>10}")
print("-" * 58)
for activity in ACTIVITIES:
    m = metrics[activity]
    print(f"{activity:<12} {m['sensitivity']:>12.3f} {m['specificity']:>12.3f} {m['support']:>10}")
print("-" * 58)
print(f"{'OVERALL ACCURACY':<12} {metrics['overall']['accuracy']:>12.3f}")

## 6. Visualizations

All figures are saved to `figures/` for inclusion in the report.

In [ ]:
os.makedirs("figures", exist_ok=True)

# Transition matrix heatmap
plot_transition_matrix(model.transmat_, activities=ACTIVITIES,
                       save_path="figures/transition_matrix.png")

# Emission matrix heatmap
plot_emission_matrix(model.emissionprob_, activities=ACTIVITIES,
                     save_path="figures/emission_matrix.png")

# Baum-Welch convergence
plot_log_likelihood(model.log_likelihoods_,
                    save_path="figures/convergence.png")

# Confusion matrix
plot_confusion_matrix(y_test_true, y_test_pred, activities=ACTIVITIES,
                      save_path="figures/confusion_matrix.png",
                      title="Confusion Matrix — Unseen Test Data")

# Decoded sequence (first test file)
first_test = test_info[0]
n_win = first_test["n_windows"]
seq_symbols = X_test_symbols[:n_win]
pred_states = model.predict(seq_symbols)
activity_to_state = {a: s for s, a in state_to_activity.items()}
true_state_indices = np.array([activity_to_state[first_test["activity"]]] * n_win)
plot_decoded_sequence(true_state_indices, pred_states, activities=ACTIVITIES,
                      save_path="figures/decoded_sequence.png")

print("All figures saved to figures/")

## 7. Transition & Emission Matrices (printed for report)

These can be copied directly into the report.

In [ ]:
import pandas as pd

# Transition matrix
trans_df = pd.DataFrame(model.transmat_, index=ACTIVITIES, columns=ACTIVITIES)
print("Transition Matrix (A):")
print(trans_df.round(3))
print()

# Emission matrix (shape: n_hidden x n_obs)
emi_df = pd.DataFrame(model.emissionprob_, index=ACTIVITIES)
print("Emission Matrix (B) — first 10 observation symbols:")
print(emi_df.iloc[:, :10].round(3))

## 8. Submission Checklist

| Item | Status |
|------|--------|
| 40 labelled CSV training files (10 per activity) | ✅ `data/train/` |
| 8 unseen test CSV files (2 per activity) | ✅ `data/test/` |
| Python notebook implementing HMM | ✅ this notebook |
| HMM from scratch (Viterbi + Baum-Welch, convergence check) | ✅ `src/hmm.py` |
| Transition matrix heatmap | ✅ `figures/transition_matrix.png` |
| Decoded sequence plot | ✅ `figures/decoded_sequence.png` |
| Confusion matrix | ✅ `figures/confusion_matrix.png` |
| Sensitivity / Specificity / Accuracy table | ✅ printed above |
| 4 data-collection videos (5 s each, face visible) | ⚠️ record separately — see `videos/` |
| Short report (4–5 pages) | 📝 see `report/` template |

### Important: Videos
The rubric awards an **immediate zero** if any of the 4 activity videos is missing.
Record one 5-second video per activity showing your full body holding the phone, with
your face visible and proper lighting. Place links in the report under Data Collection.